## Bibliotecas

In [3]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup


#### Config de logging


In [4]:
logger = logging.getLogger("realgm_scraper")
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)
logger.setLevel(logging.INFO)

REALGM_URL = "https://basketball.realgm.com/nba/players"

##### Navegação e captura de HTML (com Selenium)

In [5]:
def fetch_html_requests(url: str, timeout: int = 30) -> str:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/127.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://basketball.realgm.com/",
        "Connection": "keep-alive",
    }
    with requests.Session() as s:
        s.headers.update(headers)
        # Visita a home antes (alguns sites setam cookies)
        s.get("https://basketball.realgm.com/", timeout=timeout)
        r = s.get(url, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
        logger.info("HTML baixado via requests (%d chars).", len(r.text))
        return r.text

#### Fallback opcional via Selenium (use se o site bloquear requests)

In [6]:
def fetch_html_selenium(url: str, wait_css_selector: str = "body", timeout: int = 20) -> str:
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options as ChromeOptions
        from selenium.webdriver.chrome.service import Service
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support import expected_conditions as EC  # type: ignore
        from selenium.webdriver.support.ui import WebDriverWait
        from webdriver_manager.chrome import ChromeDriverManager
    except Exception as e:
        raise RuntimeError(
            "Selenium/WebDriver não disponíveis. Instale: "
            "`pip install selenium webdriver-manager`"
        ) from e

    chrome_opts = ChromeOptions()
    chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--window-size=1920,1080")
    chrome_opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),
                              options=chrome_opts)
    try:
        driver.get(url)
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css_selector))
        )
        html = driver.page_source
        logger.info("HTML renderizado via Selenium (%d chars).", len(html))
        return html
    finally:
        driver.quit()


#### Localiza a tabela correta (robusto para mudanças leves no layout)

In [7]:
def find_players_table(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")

    # Busca a primeira <table> que contenha um <th> com o texto "Player"
    for table in soup.find_all("table"):
        headers = [th.get_text(strip=True) for th in table.find_all("th")]
        if any(h.lower() == "player" for h in headers):
            return str(table)

    # fallback: se não achar, tenta qualquer tabela com muitas linhas
    candidate = None
    max_rows = 0
    for table in soup.find_all("table"):
        rows = len(table.find_all("tr"))
        if rows > max_rows:
            max_rows = rows
            candidate = table

    if candidate:
        logger.warning("Tabela com 'Player' não encontrada; usando maior tabela da página (linhas=%d).", max_rows)
        return str(candidate)

    raise RuntimeError("Nenhuma tabela encontrada no HTML informado.")

#####  Parse → DataFrame + limpeza leve

In [8]:
def parse_players_table(html: str) -> pd.DataFrame:
    table_html = find_players_table(html)
    dfs = pd.read_html(table_html)
    if not dfs:
        raise RuntimeError("pd.read_html não retornou tabelas para o fragmento selecionado.")
    df = dfs[0]

    # Normalizações comuns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(map(str, c)).strip() for c in df.columns]
    df.columns = [c.strip() for c in df.columns]
    df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]

    # Padroniza alguns nomes usuais (ajuste se necessário)
    rename_map = {
        "Player": "player",
        "Team": "team",
        "Pos": "pos",
        "Ht": "height",
        "Wt": "weight",
        "DOB": "dob",
        "From": "from",
        "To": "to",
        "Yrs": "years",
        "College": "college",
        "Country": "country",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # Conversões leves (só se existirem as colunas)
    for col in ("years", "weight"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

##### # Orquestrador: tenta requests, faz fallback opcional p/ Selenium

In [9]:
def get_realgm_players(
    url: str = REALGM_URL,
    save_html: Optional[Path | str] = "realgm_players.html",
    use_selenium_fallback: bool = True,
) -> pd.DataFrame:
    try:
        html = fetch_html_requests(url)
    except Exception as e_req:
        logger.warning("Requests falhou (%s).", e_req)
        if not use_selenium_fallback:
            raise
        logger.info("Tentando fallback via Selenium…")
        html = fetch_html_selenium(url)

    if save_html:
        Path(save_html).write_text(html, encoding="utf-8")
        logger.info("HTML persistido em: %s", Path(save_html).resolve())

    df = parse_players_table(html)
    logger.info("DataFrame final: %s linhas x %s colunas", df.shape[0], df.shape[1])
    return df


##### Execucao

In [10]:
df = get_realgm_players(
    url=REALGM_URL,
    save_html="realgm_players.html",
    use_selenium_fallback=True
)

# ID sequencial com 5 zeros à esquerda (00001, 00002, ...)
df = df.reset_index(drop=True)
df.insert(0, "player_id", [str(i).zfill(5) for i in range(1, len(df) + 1)])

2025-10-17 12:10:20,344 | INFO | HTML baixado via requests (624032 chars).
2025-10-17 12:10:20,348 | INFO | HTML persistido em: C:\Users\henri\OneDrive\Documents\Codigo\Basketanalysis\scraping\times\realgm_players.html
C:\Users\henri\AppData\Local\Temp\ipykernel_19644\3090488327.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(table_html)
2025-10-17 12:10:20,994 | INFO | DataFrame final: 589 linhas x 11 colunas


In [11]:
df

,player_id,#,player,pos,HT,WT,Age,Current Team,YOS,Pre-Draft Team,Draft Status,Nationality
0,00001,8.0,Precious Achiuwa,SF,6-8,243,26,Miami Heat,5,Memphis,2020 Rnd 1 Pick 20,Nigeria
1,00002,12.0,Steven Adams,C,6-11,265,32,Houston Rockets,12,Pittsburgh,2013 Rnd 1 Pick 12,New Zealand
2,00003,13.0,Bam Adebayo,C,6-9,255,28,Miami Heat,8,Kentucky,2017 Rnd 1 Pick 14,United States
3,00004,30.0,Ochai Agbaji,SF,6-5,215,25,Toronto Raptors,3,Kansas,2022 Rnd 1 Pick 14,United States
4,00005,7.0,Santi Aldama,C,7-0,215,24,Memphis Grizzlies,4,Loyola (MD),2021 Rnd 1 Pick 30,Spain
...,...,...,...,...,...,...,...,...,...,...,...,...
584,00585,17.0,Jahmir Young,PG,6-0,185,25,Miami Heat,1,Maryland,"2024 NBA Draft, Undrafted",United States
585,00586,11.0,Trae Young,PG,6-2,164,27,Atlanta Hawks,7,Oklahoma,2018 Rnd 1 Pick 5,United States
586,00587,3.0,Chris Youngblood,SG,6-4,221,23,Oklahoma City Thunder,0,Alabama,"2025 NBA Draft, Undrafted",United States
587,00588,44.0,Rocco Zikarsky,C,7-3,227,20,Minnesota Timberwolves,0,Brisbane (Australia),2025 Rnd 2 Pick 15,Australia


In [15]:
df.to_csv("../../basket_dbt/seeds/storage/raw/players.csv", index=False)